## Regressão Logística

A [Regressão Logística](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) é um modelo de classificação supervisionada amplamente utilizado em problemas binários.

O modelo funciona a partir de uma combinação linear das variáveis de entrada, produzindo um valor que é transformado em probabilidade por meio da função logística (sigmoid), resultando em valores entre 0 e 1.

### Hiperparâmetros utilizados

- **C**: controla a regularização do modelo.
  - Valores menores (ex: 0.01) → modelo mais simples (mais regularização)
  - Valores maiores (ex: 10) → modelo mais complexo (menos regularização)
  - Os valores foram escolhidos em escala logarítmica para explorar diferentes ordens de magnitude.

- **solver**: algoritmo de otimização utilizado para encontrar os coeficientes do modelo.
  - `lbfgs`: bom desempenho geral, especialmente para datasets maiores (default)
  - `liblinear`: eficiente para problemas binários e datasets menores

### Estratégia

Foi utilizado o GridSearchCV com validação cruzada (5 folds) para identificar a melhor combinação de hiperparâmetros, utilizando a métrica ROC AUC, adequada para problemas com classes desbalanceadas.

In [1]:
import pandas as pd
import sys
import os

# add raiz do projeto
sys.path.append(os.path.abspath(".."))

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

from preprocessing.main_preprocessing import load_and_preprocess
from utils.experiment_logger import save_results

## BASELINE

In [2]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

smote_options = [False, True]

results = []

for scenario in scenarios:
    for use_smote in smote_options:

        print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

        X_train, X_test, y_train, y_test = load_and_preprocess(
            "../predictive_models/scrdata_202505.csv",
            scenario=scenario,
            use_smote=use_smote
        )

        model = LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            solver="lbfgs"
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        results.append({
            "model": "LogisticRegression",
            "scenario": scenario,
            "smote": use_smote,
            "roc_auc": roc_auc_score(y_test, y_proba),
            "f1": f1_score(y_test, y_pred),
            "accuracy": accuracy_score(y_test, y_pred),
            "n_features": X_train.shape[1],
            "phase": "baseline",
            "timestamp": pd.Timestamp.now()
        })

save_results(results, file_path="../results/experiments.csv")

df_result = pd.DataFrame(results)

display(df_result)


🔎 Scenario: sem_submodalidade | SMOTE: False

🔎 Scenario: sem_submodalidade | SMOTE: True

🔎 Scenario: submodalidade_agrupada | SMOTE: False

🔎 Scenario: submodalidade_agrupada | SMOTE: True

🔎 Scenario: submodalidade_engineered | SMOTE: False

🔎 Scenario: submodalidade_engineered | SMOTE: True


,model,scenario,smote,roc_auc,f1,accuracy,n_features,phase,timestamp
0,LogisticRegression,sem_submodalidade,False,0.695678,0.442656,0.562730,67,baseline,2026-04-09 22:03:19.911033
1,LogisticRegression,sem_submodalidade,True,0.766062,0.719562,0.688867,67,baseline,2026-04-09 22:04:00.640689
2,LogisticRegression,submodalidade_agrupada,False,0.770094,0.722598,0.692225,97,baseline,2026-04-09 22:04:27.997263
3,LogisticRegression,submodalidade_agrupada,True,0.768670,0.721764,0.690746,97,baseline,2026-04-09 22:05:09.818408
4,LogisticRegression,submodalidade_engineered,False,0.695678,0.442656,0.562730,67,baseline,2026-04-09 22:05:22.348147
5,LogisticRegression,submodalidade_engineered,True,0.766062,0.719562,0.688867,67,baseline,2026-04-09 22:06:02.208107


## GRIDSEARCH

In [4]:
X_train, X_test, y_train, y_test = load_and_preprocess(
    "../predictive_models/scrdata_202505.csv",
    scenario="sem_submodalidade",
    use_smote=False
)


param_grid_lr = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["lbfgs"]
}

grid_lr = GridSearchCV(
    estimator=LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    param_grid=param_grid_lr,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_lr.fit(X_train, y_train)

print("Best parameters:", grid_lr.best_params_)
print("Best ROC AUC (CV):", grid_lr.best_score_)


best_lr = grid_lr.best_estimator_

y_pred = best_lr.predict(X_test)
y_proba = best_lr.predict_proba(X_test)[:, 1]


tuning_result = [{
    "model": "LogisticRegression",
    "scenario": "sem_submodalidade",
    "smote": False,
    "phase": "tuning_cv",
    "roc_auc": grid_lr.best_score_,
    "f1": None,
    "accuracy": None,
    "best_params": str(grid_lr.best_params_),
    "timestamp": pd.Timestamp.now()
}]

final_result = [{
    "model": "LogisticRegression",
    "scenario": "sem_submodalidade",
    "smote": False,
    "phase": "test",
    "roc_auc": roc_auc_score(y_test, y_proba),
    "f1": f1_score(y_test, y_pred),
    "accuracy": accuracy_score(y_test, y_pred),
    "best_params": str(grid_lr.best_params_),
    "timestamp": pd.Timestamp.now()
}]


save_results(tuning_result, file_path="../results/model_results.csv")
save_results(final_result, file_path="../results/model_results.csv")


df_result = pd.DataFrame(final_result)
display(df_result)

Best parameters: {'C': 1, 'solver': 'lbfgs'}
Best ROC AUC (CV): 0.7717980452833051


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,LogisticRegression,sem_submodalidade,False,test,0.695678,0.442656,0.56273,"{'C': 1, 'solver': 'lbfgs'}",2026-04-09 22:07:49.514070
